In [ ]:
!git clone https://github.com/jackzengh/grpo.git
%cd grpo
!pip install -r requirements-gpu.txt

: 

In [30]:
!nvidia-smi | grep -i "CUDA Version"
!ls /usr/local/lib/python3.12/dist-packages/ | grep -i nvidia_cuda_runtime || echo "no cuda-runtime wheel"
import torch; print("torch:", torch.__version__, "| torch sees CUDA:", torch.version.cuda, "| GPU usable:", torch.cuda.is_available())

| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
nvidia_cuda_runtime-13.3.29.dist-info
nvidia_cuda_runtime_cu12-12.8.90.dist-info
torch: 2.11.0+cu128 | torch sees CUDA: 12.8 | GPU usable: True


In [35]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"+ os.environ.get("LD_LIBRARY_PATH", "")
)
print(os.environ["LD_LIBRARY_PATH"])

/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:/usr/lib64-nvidia


In [31]:
import os
from pathlib import Path

SCRATCH = Path.home() / "scratch"
os.environ["HF_HOME"] = str(SCRATCH / "hf_home")

In [32]:
import gc
import re
import time
from typing import Any, Dict, List, Tuple, Union

import deepspeed
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from deepspeed import DeepSpeedEngine
from tqdm import trange
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel
from vllm import LLM, SamplingParams

import wandb
from utils import (
	compute_token_log_probs,
	extract_boxed_answer,
    dump_episodes,
    evaluate_on_test_set,
    find_free_port,
    find_last_checkpoint,
    prepare_model_inputs,
    load_model_into_vllm
)

# Needed to stop DeepSpeed from complaining
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = str(find_free_port())
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-3B"
MODEL_CHAT_NAME = MODEL_NAME + "-Instruct"

# Dataset configuration
DATASET_NAME = "EleutherAI/hendrycks_math"

### Setting up our variables

- Number of iterations = how many gradient updates we'll do total
- Episodes per iteration = our **effective batch size**, how many episodes before we step

**Actual batch size x grad_accum steps = episodes per iteration** _(each episode here is a completion instead of a finished game)_

- Per-device batch size = microbatch size, our true batch size
- Generations per sample = GRPO group size

Grad accum steps = how many forward passes on a GPU we do before we do an optimizer step.

Note this is just episodes per iteration divided by per-device batch size!


In [ ]:
# Total number of training iterations
NUM_ITERATIONS = 1000
# Number of episodes to collect per iteration for training
EPISODES_PER_ITERATION = 64
# Number of responses to generate for each input prompt (i.e. group size in GRPO)
GENERATIONS_PER_SAMPLE = 4

# Controls how much the policy can deviate from the reference model
KL_COEFFICIENT = 0.001

# Training hyperparameters
# Batch size for each GPU device during training
PER_DEVICE_BATCH_SIZE = 4
# Learning rate for model updates
LEARNING_RATE = 1e-6

# Sampling parameters
# Maximum number of tokens to generate in each response
MAX_RESPONSE_TOKENS = 1024
# Controls randomness in generation (higher = more random)
TEMPERATURE = 1.0
# Nucleus sampling parameter (1.0 = disabled)
TOP_P = 1.0
# Top-k sampling parameter (-1 = disabled)
TOP_K = -1  # no top k

# DeepSpeed configuration
# DeepSpeed config for the policy model
deepspeed_config = {
	"bf16": {"enabled": True},
	"zero_optimization": {"stage": 2, "overlap_comm": False},
	"train_batch_size": EPISODES_PER_ITERATION, # optimizer step
	"train_micro_batch_size_per_gpu": PER_DEVICE_BATCH_SIZE, # per GPU allocated batch size
	"gradient_accumulation_steps": EPISODES_PER_ITERATION // PER_DEVICE_BATCH_SIZE, # increase our effective batch size by adding up gradients across devices
	"gradient_clipping": 1.0,
	"optimizer": {
		"type": "AdamW",
		"params": {
			"lr": LEARNING_RATE,
			"betas": (0.9, 0.999),
			"eps": 1e-8,
			"weight_decay": 0.0,
			"torch_adam": True,
		},
	},
}
# DeepSpeed config for the reference model that we use to compute KL divergence
ref_deepspeed_config = {
	"bf16": {"enabled": True},
	# Note that we don't train the reference model
	# These are just for compatibility with DeepSpeed.
	"train_batch_size": EPISODES_PER_ITERATION,
	"train_micro_batch_size_per_gpu": PER_DEVICE_BATCH_SIZE,
	"gradient_accumulation_steps": EPISODES_PER_ITERATION // PER_DEVICE_BATCH_SIZE,
}

RUN_NAME = "r1-zero"
EXP_DIR = SCRATCH / "deepseek_r1z_hackathon" / RUN_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Logs and Checkpoints will be saved to: {EXP_DIR}")

Logs and Checkpoints will be saved to: /Users/jackzheng/scratch/deepseek_r1z_hackathon/r1-zero


In [ ]:
SYSTEM_MESSAGE = (
	"You are a helpful assistant. You first think about the reasoning process in the mind "
	"and then provide the user with the answer."
)
PROMPT_TEMPLATE = (
	"{problem}\n"
	"Show your work in <think> </think> tags. And return the final answer in "
	"<answer> </answer> tags, for example <answer>42</answer>."
)

### Tokenizing + loading dataset


In [ ]:
# we use the chat model because it has apply chat template, which is the format we want to converse with this LLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHAT_NAME)
EOS_TOKEN_ID = AutoTokenizer.from_pretrained(MODEL_NAME).eos_token_id
EOS_TOKEN = tokenizer.convert_ids_to_tokens(EOS_TOKEN_ID)

# Dataloader to load MATH dataset
def data_loader(example: Dict[str, Any]):

	prefix = [
		{"role": "system", "content": SYSTEM_MESSAGE},
		{"role": "user", "content": PROMPT_TEMPLATE.format(problem=example["problem"])}, # load the problem set in
		{"role": "assistant", "content": "Let me solve this step by step.\n<think>"},
	]
	input_ids = tokenizer.apply_chat_template(
		prefix, tokenize=True, continue_final_message=True # we want to model to generate its own answer
	)
	prompt = tokenizer.decode(
		input_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
	)
	return {"prompt": prompt, "input_ids": input_ids, "answer": extract_boxed_answer(example["solution"])}

# load the MATH dataset that contains problems across subjects
MATH_SUBJECTS = [
	 "algebra", "counting_and_probability", "geometry", "intermediate_algebra", "number_theory", "prealgebra", "precalculus",
]

def load_math_split(split: str): 
	data = [load_dataset(DATASET_NAME, name=s, split=split) for s in MATH_SUBJECTS] # assemble array 
	return concatenate_datasets(data)

train_dataset = load_math_split('train').map(data_loader).filter(lambda ex: ex["answer"] is not None)
test_dataset = load_math_split('test').map(data_loader).filter(lambda ex: ex["answer"] is not None)

NameError: name 'AutoTokenizer' is not defined

In [ ]:
len(train_dataset), len(test_dataset)

test_dataset[0]

NameError: name 'train_dataset' is not defined

### Building the reward function

1. Formatting rewards - is it returning with <think> and <answer> tokens
2. Answer rewards - is the answer actually right? Use extract_from_boxed to extract final solution. No intermediate rewards. Compre using hf/math_verify
3. Compute total reward - sum both of the above together, for logging purposes keep both


In [ ]:
from math_verify import parse, verify

def format_reward_fn(completion: str): 
	try:
		# add synthetic <think> as its already part of the prompt and prefilled 
		# for the assistant to more easily match the regex
		completion = "<think>" + completion

		# Strip EOS token if present
		if completion.endswith(EOS_TOKEN):
			completion = completion[:-len(EOS_TOKEN)]

		# Check if the format is correct
		# Pattern means:
		# 1) <think>...contents not including other <think> tags...</think>
		# 2) \n
		# 3) <answer>...anything...</answer>  
		regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\/think>\n<answer>([\s\S]*?)<\/answer>$"
		match = re.search(regex, completion, re.DOTALL)

		if match is None or len(match.groups()) != 2:
			# Format is incorrect
			return 0.0
		else:
			# Extract the content inside <answer>...</answer>
			answer_content = match.group(2).strip()

			# Check if answer content matches the allowed pattern
			if not re.match(allowed_pattern, answer_content):
				# If it doesn't match, reward is 0.5
				return 0.5
			# If both format and pattern are correct, reward is 1
			return 1.0
	except Exception:
		# Any error leads to 0 reward
		return 0.0

def answer_reward_fn(completion: str, prompt: str, answer: str): 

	try:
		# Pull the text inside the LAST <answer>...</answer> the model wrote.
		matches = re.findall(r"<answer>([\s\S]*?)<\/answer>", completion)
		if not matches:
			return 0.0
		model_answer = matches[-1].strip() # get the last answer match
		if model_answer == "" or answer is None: # if empty, return 0 reward
			return 0.0

		# math_verify parses both sides into symbolic math  objects and checks equivalence (so \frac{1}{2}, 0.5, and 2/4 all count as equal).
		# verify(gold, target) -> True if equal. We wrap the gold answer in $...$ so the parser treats it as LaTeX math.
		gold = parse(f"${answer}$")
		pred = parse(model_answer)
		return 1.0 if verify(gold, pred) else 0.0
	except Exception:
		return 0.0

def compute_reward(completion: str, sample: Dict[str, Any]):

	format_reward = format_reward_fn(completion)
	answer_reward = answer_reward_fn(completion, sample['prompt'], sample['answer'])
	reward = format_reward + answer_reward

	metrics = { # for graphing later on
		"format_reward": format_reward,
		"answer_reward": answer_reward,
		"reward": reward,
	}

	return reward, metrics

### Calculate advantage

The only role of this funciton is to calculate the reward, advantage for a series of (query, completion) pairs

Does this for many completions

- sample is taken from the training data set
- all_generations is taken from a completion by the vLLM inference model (i.e. the actor)
- all_finish_reasons is also computed by the actor and handed out by vLLM in outputs.outputs.finish_reason


In [ ]:
# each response is a list of dictionaries, each dictionary containing {prompt: str, answer: str, etc.}
def calculate_advantage(sample: List[Dict[str: Any]], all_generations: List[List[int]], all_finish_reasons: List[str]):

	assert len(all_generations) == len(all_finish_reasons)
	assert len(all_generations) == len(samples) * GENERATIONS_PER_SAMPLE

	groups = [
		# list that defines where each completion (non-unique prompt) lives
		list(range(i, i + GENERATIONS_PER_SAMPLE)) for i in range(0, len(all_generations), GENERATIONS_PER_SAMPLE)
	]

	all_query_token_ids, all_responses_token_ids, all_advantages = [], [], []

	stats = {
		"response_lengths": [],
		"rewards": [],
		"non_stop_rate": [],
	}

	# sample is a list of dicts - the prompt, answer, etc. tuples, for which we've sampled multiple completions for 
	for sample, group_indices in zip(sample, group):
		# where group_indices is [1,2,3,4] if GENERATIONS_PER_SAMPLE = 4 for example
		finish_reasons = [all_finish_reasons[i] for i in group_indices] # pluck out reasons from that group
		response_token_ids = [all_generations[i] for i in group_indices] # pluck out response tokens, returns array
		responses = tokenzier.batch_decode(response_token_ids, skip_special_tokens=False) # returns array but decoded with [completion1, completion2, etc.]

		rewards_raw = [compute_reward(resp, sample) for resp,sample in zip(responses, sample)] # compare the response with the golden, return array of rewards for each
		rewards, reward_metrics = zip(*rewards_raw) # will return as 'rewards, metrics'

		rewards = np.array(rewards)
		response_advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4) # normalize to get advantages per token

		advantages = [
			[resp_adv] * len(resp) for resp_adv, resp in zip(response_advantages, response_token_ids) # this is list multiplication
			# so we get [7,7,7,7,...] where the # of 7s is the length of our list, now we want to match those with the tokens
		]

		all_query_token_ids.extend([sample["input_ids"]] * GENERATIONS_PER_SAMPLE)
		all_responses_token_ids.extend(response_token_ids)
		all_advantages.extend(advantages)

		stats["rewards"].extend(rewards)
		stats["non_stop_rate"].extend([fr != "stop" for fr in finish_reasons])
		stats["response_lengths"].extend([len(ids) for ids in response_token_ids])
		for rm in reward_metrics:
			for k, v in rm.items():
				stats.setdefault(f"reward_metrics/{k}", []).append(v)

	episodes = {
		"all_query_token_ids": all_query_token_ids,
		"all_response_token_ids": all_responses_token_ids,
		"all_advantages": all_advantages,
	} 

	return episodes, stats

TabError: inconsistent use of tabs and spaces in indentation (<string>, line 21)

### Compute loss

Takes in policy + ref model and batch that includes input_tokens + labels

- use compute_token_log_probs that calculates per token, the log_probs of generating it with the new model
- we then find the KL divergence between the new vs ref model
- using K3 divergence estimation: http://joschu.net/blog/kl-approx.html
- return the loss, entropy, KL penalty


In [ ]:
def compute_loss(
	policy_model: DeepSpeedEngine | PreTrainedModel,
	reference_model: DeepSpeedEngine | PreTrainedModel,
	batch: Dict[str, torch.Tensor],
	total_response_len: int,
):
	input_ids = batch["input_ids"]  # [batch_size, seq_len]
	attention_mask = batch["attention_mask"]  # [batch_size, seq_len]
	labels = batch["labels"]  # [batch_size, seq_len]
	advantages = batch["advantages"]  # [batch_size, seq_len]

	model_inputs = {
		"input_ids": input_ids,
		"attention_mask": attention_mask,
		"labels": labels,
	}

	labels_mask = (labels[..., 1:] != -100).float()  # take the last dimension and grab only from first to last 
	# this is becuase we want to compare position 0 to position 1, etc. - [batch_size, seq_len-1]
	# MASK: set all labels that are not equal to -100 to True, so that means we will train on them
	# RECALL: labels = [ignore_index] * len(query) + response + [ignore_index] * (max_seq_len-seq_len)

	with torch.no_grad():
		ref_logps = compute_token_log_probs(
			reference_model, model_inputs, TEMPERATURE
		)  # [batch_size, seq_len-1]

	# need to recompute log_probs since we want to track gradients
	logps = compute_token_log_probs(policy_model, model_inputs, TEMPERATURE)  # [batch_size, seq_len-1]

	# Approximation of the KL divergence - K3 divergence
	# 
	kl_penalty = torch.exp(ref_logps - logps) - (ref_logps - logps) - 1  # [batch_size, seq_len-1]
	
	# lgprobs is [batch, seq_len-1]
	# ref_lgprobs is [batch, seq_len-1]

	kl_penalty = kl_penalty * labels_mask  # [batch_size, seq_len-1] element-wise multiply that zeros out the padding tokens + query
	# would not make sense to calculate KL divergence btwn ref and policy models on zero'ed out tokens

	entropy = -logps.sum() / labels_mask.sum()  # scalar

	# 
	policy_loss = -logps * advantages[..., 1:]  # [batch_size, seq_len-1]
	policy_loss = policy_loss * labels_mask  # [batch_size, seq_len-1]

	# 
	loss = (policy_loss + KL_COEFFICIENT * kl_penalty).sum() / total_response_len  

	metrics = {
		"policy_loss": policy_loss.sum().item() / total_response_len,
		"kl_penalty": kl_penalty.sum().item() / total_response_len,
		"entropy": entropy.item() / total_response_len,
	}

	return loss, metrics

### Initialize main and reference models


In [ ]:
policy_model = AutoModelForCausalLM.from_pretrained(
	MODEL_NAME,
	attn_implementation="flash_attention_2",
	torch_dtype=torch.bfloat16,
	device_map=0,
)
reference_model = AutoModelForCausalLM.from_pretrained(
	MODEL_NAME,
	attn_implementation="flash_attention_2",
	torch_dtype=torch.bfloat16,
	device_map=0,
)
policy_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Initialize DeepSpeed engines for efficient distributed training
policy_model, *_ = deepspeed.initialize(
    model=policy_model,
    config=deepspeed_config,
    model_parameters=policy_model.parameters(),
)
reference_model, *_ = deepspeed.initialize(
    model=reference_model,
    config=ref_deepspeed_config,
)

reference_model.module.cpu()

############################################

# Initialize vLLM (Inference) engine

############################################

# This is a copy of our policy model that we'll update after every iteration (set # of episodes)
inference_engine = LLM(
	model=MODEL_NAME,
	skip_tokenizer_init=False,
	gpu_memory_utilization=0.2,
	enable_prefix_caching=True,
	swap_space=1,
	scheduling_policy="fcfs",
	dtype=torch.bfloat16,
	max_model_len=2048,
	enable_sleep_mode=True,
)

# Wandb for logging

.init(
	project="grpo-math",
	name=RUN_NAME,
	config={
	"model_name": MODEL_NAME,
	"learning_rate": LEARNING_RATE,
	"num_iterations": NUM_ITERATIONS,
	"episodes_per_iteration": EPISODES_PER_ITERATION,
	"rollouts_per_episode": GENERATIONS_PER_SAMPLE,
	"kl_coefficient": KL_COEFFICIENT,
	"temperature": TEMPERATURE,
	},
)

### Load last checkpoint

Returns the location of the latest model weights and optimizer states


In [ ]:
# Load checkpoint if it exists
begin_iter = 0
ckpt_path, ckpt_iter = find_last_checkpoint(EXP_DIR)
if ckpt_path is not None:
	print(f"Resuming from checkpoint {ckpt_path} at iteration {ckpt_iter}")
	out = policy_model.load_checkpoint(ckpt_path / "deepspeed")
	if out is None:
		raise RuntimeError(f"Failed to load checkpoint {ckpt_path}")
	begin_iter = ckpt_iter + 1
	load_model_into_vllm(policy_model, inference_engine)

### Training Code

1. Evaluate current model on test set
2. Sample rollouts for the prompt
3. Calculate sequence reward
4. Compute policy gradient loss
5. Step the learner policy model
6. Update the actor model on vLLM
7. Log the changes in wandb
8. Checkpoint the model + optimizer states

Implement sleeping stage 2 - stage 1 evicts the KV cache, stage 2 evicts the model weights and KV cache

- we'll be using stage 2 since we won't need either for the next prompt rollout + new actor model


In [ ]:
for iteration in trange(NUM_ITERATIONS): # tqdm shorthand
	print(f"Iteration {iteration}/{NUM_ITERATIONS}")

	metrics = {} # for logging!
	
	# do evals on the current actor model
	eval_states = None 
	if iteration % 25 == 0: 
		print("Evaluating on eval set...")
		eval_episodes, eval_states = evaluate_on_test_set(
			inference_engine=inference_engine,
            test_dataset=test_dataset,
            tokenizer=tokenizer,
            eos_token=EOS_TOKEN,
            eval_sampling_params=SamplingParams(
                temperature=0.3,
                max_tokens=1024,
                n=1,
                detokenize=False, # we're passing in tokens!
                stop_token_ids=[EOS_TOKEN_ID],
            ),
            reward_func=lambda completion, sample: compute_reward(
                completion, sample
            ), 
		)
		eval_episode_table = dump_episodes(
            episodes=eval_episodes,
            episodes_stats=eval_stats,
            exp_dir=EXP_DIR,
            tokenizer=tokenizer,
            iteration=iteration,
            is_eval=True,
        )
        wandb.log({"eval/episodes": eval_episode_table, "iteration": iteration})

	# sample training batch
	# how many samples should we do (i.e. unique prompts)
	# more simply COMPLETIONS_PER_ITERATION // COMPLETIONS_PER_PROMPT
	num_samples = EPISODES_PER_ITERATION // GENERATIONS_PER_SAMPLE

	# returns num_samples random integers in a 1D array, no replacement so samples uniquely
	indices = np.random.choice(
		len(train_dataset), size=num_samples, replace=False
	)

	# pluck out training samples - which are dicts
	samples = train_dataset.select(indices)

	# sample responses
	outputs = inference_engine.generate(
		prompt_token_ids=samples[input_ids] # [[tokenids], [tokenids], ...]
		sampling_params=SamplingParams(
			n=GENERATIONS_PER_SAMPLE,
			temperature=TEMPERATURE,
			top_k=TOP_K,
			top_p=TOP_P,
			max_tokens=MAX_RESPONSE_TOKENS,
			detokenize=False,
			stop_token_ids=[EOS_TOKEN_ID]
		)
	)

	"""
	outputs is of shape

	RequestOutput(
    request_id='...',
    prompt='Hello, my name is',
    prompt_token_ids=[...],
    outputs=[
        CompletionOutput1(
            index=0,
            text='...',
            token_ids=[...],
            cumulative_logprob=...,
            logprobs=...,
            finish_reason='...',
            stop_reason=...
        )
		CompletionOutput2(
            index=0,
            text='...',
            token_ids=[...],
            cumulative_logprob=...,
            logprobs=...,
            finish_reason='...',
            stop_reason=...
        )
    ],
    finished=True,
    metrics=...
)

	"""
	# outputs is an array of the unique outputs, then within each is the multiple completions per sample
	all_generations = [list(g.token_ids) for out in outputs for g in out.outputs]
	all_finish_reasons = [g.finish_reason for out in outputs for g in out.outputs]
	inference_engine.sleep(1)

	print(f"Generated {len(all_generations)} responses")
	gc.collect() # garbage collection
	torch.cuda.empty_cache() # free up GPU memory that we just cleared from deleting the KV cache from HBM
	time.sleep(1)

	episodes, episodes_stats = calculate_advantage(
		samples,
		all_generations,
		all_finish_reasons,
	) # episodes now holds the generations, associated prompts and token by token computed advantages 

	"""
	return structure looks like this: 

	episodes = {
		"all_query_token_ids": all_query_token_ids,
		"all_response_token_ids": all_responses_token_ids,
		"all_advantages": all_advantages,
	} 
	"""

	for k, v in episodes_stats.items(): 
		metrics.setdefault(k, []).append(v) # append response_lengths, rewards, non_stop_rates

	### LOGGING

	episode_table = dump_episodes(
		episodes,
		episodes_stats,
		EXP_DIR,
		tokenizer,
		iteration,
	) # returns table structured like wandb - where each column is "query", "response", "reward", "response_length"

	### TRAINING
	# now that we have advantages for the generation tokens, with the associated query tokens we need to update our model
	# this means computing the loss, which is complex because we're using 2 different models - actor & learner

	# prep the data for loss computation - that is turn these query+completions into labels and make the attention + labels masks
	# attention masking is for padding
	# labels masking is ensuring we don't pull log_probs for query / padding tokens - not what we want model to be training on
	model_inputs = prepare_model_inputs(
		query_token_ids=episodes["all_query_token_ids"],
		response_token_ids=episodes["all_response_token_ids"],
		advantages=episodes["all_advantages"],
		device="cuda"
	)

	# Prep the models!!
	policy_model.train()
	
	# ref model which we use ONLY in loss computation for KLD
	reference_model.module.cuda() # bring ref model that we saved to cpu to gpu, module needed if ref_model is wrapped in DDP, etc.
	reference_model.eval() # turns off dropout, stops gradient tracking

	# Find response length for us to normalize by later in the loss fn
	total_response_len = (model_inputs["labels"] != -100).sum().item() # creates T/F array to find how many tokens in labels are real tokens, then counts them

	# recall per device batch size is microbatch size 
	for i in trange(0, EPISODES_PER_ITERATION, PER_DEVICE_BATCH_SIZE, desc="Gradient Accumulation"):
		
		# model_items is a dictionary of inputs: {[input_ids],[labels], [labels_mask], etc.}
		# from each of these arrays, e.g. input_ids, we want to pull token_ids in the respective indices for this microbatch size
		batch = {
			k: v[i: i + PER_DEVICE_BATCH_SIZE] for k,v in model_inputs.items() 
		}

		"""
		batch = {
			input_ids: ["input_ids"] <- an array of input_ids of per_device_batch_size, i.e. the microbatch size
			attention_mask: ["attention_mask"]
			labels: ["labels"]
			advantages: ["advantages"]
			labels_mask: ["labels_mask"]
		}
		
		"""

		# Compute loss
		loss, loss_metrics = compute_loss(
			policy_model,
			reference_model,
			batch,
			total_response_len,
		)

		# Track metrics
		metrics.setdefault("loss", []).append(loss.item())
		
		# check our gradients are healthy
		grad_norm = policy_model.get_global_grad_norm() # add sqrt of all gradients^2 across all parameters
		if grad_norm is not None:
			grad_norm = grad_norm.item()
		metrics.setdefault(grad_norm, []).append(grad_norm)
		
		# save loss_metrics to metrics as well
		for k, v in loss_metrics.items():
			metrics.setdefault(k, []).append(v.item() if isinstance(v, torch.Tensor) else v)

		# backprop! 
		policy_model.backward(loss, scale_wrt_gas=False) # prevent division by grad_accum_steps

		# free up memory, deletes instantly
		del loss, loss_metrics

		# 
		if policy_model.is_gradient_accumulation_boundary():
			reference_model.module.cpu()

		policy_model.step()

	# now after we've finished with our optimizer step, i.e. one iteration - we update our actor model and log our iteration results

	#########################################################
	# Update inference engine weights
	#########################################################
	
	gc.collect() 
	torch.cuda.empty_cache() # clean up ref model from HBM
	time.sleep(1)

	inference_engine.wake_up()
	load_model_into_vllm(policy_model, inference_engine) # load the policy_model into vllm

	gc.collect()
	torch.cuda.empty_cache()
	time.sleep(1)


	#########################################################
	# Log metrics
	#########################################################

	train_metrics = {
		k: np.mean(v) for k, v in metrics.items() if None not in v # checks that every field exists
	}
	train_metrics["learning_rate"] = policy_model.get_lr()[0]
	logs = {
		"iteration": iteration,
		f"episodes/iter_{iteration:06d}": episode_table,
		
		# ** unpacks the dictionary into individual items so it gets saved as individual entries in the logs dict
		**{f"train/{k}": v for k,v in train_metrics.items()}, # save in train/ all our train metrics like loss, etc.
	}

	if eval_stats is not None: 
		eval_metrics = {k: np.mean(v) for k, v in eval_stats.items() if None not in v}
		logs.update({f"eval/{k}": v for k, v in eval_metrics.items()})

		"""
		eval_stats
		
		metrics = {
			"response_lengths": [],
			"rewards": [],
			"non_stop_rate": [],
		}
		"""
	wandb.log(logs)

	selected_keys = [
        "train/kl_penalty",
        "train/rewards",
        "train/reward_metrics/format_reward",
        "train/reward_metrics/equation_reward",
        "eval/rewards",
        "eval/reward_metrics/format_reward",
        "eval/reward_metrics/equation_reward",
    ]
	selected_metrics = {k:logs[k] for k in selected_keys if k in logs}
	print(f"Key metrics: {selected_metrics}")

	# checkpoint our policy model every 50 iterations 
	if iteration % 50 == 0 and iteration != 0: 
		policy_model.save_pretrained(str(EXP_DIR / "checkpoints" / f"ckpt_{iteration:06d}" / "hf_model"))
		policy_model.save_checkpoint(str(EXP_DIR / "checkpoints" / f"ckpt_{iteration:06d}" / "hf_model"))

### Improvements I would make

- Cosine decay on lr scheduler
- Flash attention 3
- CISPO pg loss
- GPTQ for more efficient training
  - we were using torch.long and floats, change to bfloat16 at least
